In [50]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve

In [51]:
# 1. Load Data
print("Loading data...")
train_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\train.csv")
test_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\test.csv")

X_train = train_df.drop(columns=['CoilID', 'Y'])
y_train = train_df['Y']
X_test = test_df.drop(columns=['CoilID'])
test_ids = test_df['CoilID']

Loading data...


In [52]:
# 2. Preprocessing
print("Imputing and scaling...")
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc = scaler.transform(X_test_imp)

weight_ratio = 25

Imputing and scaling...


In [53]:
# 3. Initialize Models
models = {
    'RandomForest': RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=500, scale_pos_weight=weight_ratio, random_state=42, n_jobs=-1),
    'CatBoost': CatBoostClassifier(iterations=500, scale_pos_weight=weight_ratio, verbose=0, random_state=42)
}

In [54]:
# 4. Out-Of-Fold (OOF) Cross Validation for Safe Thresholding
print("Running Stratified K-Fold CV for Ensemble...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Matrix to hold OOF probabilities: [number of rows, number of models]
oof_probs = np.zeros((len(X_train), len(models)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sc, y_train)):
    print(f"--- Fold {fold + 1} ---")
    X_tr, y_tr = X_train_sc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_sc[val_idx], y_train.iloc[val_idx]
    
    for i, (name, model) in enumerate(models.items()):
        model.fit(X_tr, y_tr)
        oof_probs[val_idx, i] = model.predict_proba(X_val)[:, 1]

# Average the OOF predictions across all models
ensemble_oof_probs = oof_probs.mean(axis=1)

Running Stratified K-Fold CV for Ensemble...
--- Fold 1 ---
--- Fold 2 ---
--- Fold 3 ---
--- Fold 4 ---
--- Fold 5 ---


In [55]:
# 5. Calculate Threshold on OOF Data
precisions, recalls, thresholds = precision_recall_curve(y_train, ensemble_oof_probs)

# We must have 100% recall. Find the highest threshold that still catches everything.
valid_mask = recalls[:-1] >= 0.999 
best_threshold = thresholds[valid_mask][-1] if valid_mask.any() else 0.05
final_threshold = max(0.01, best_threshold - 0.005) # Tiny safety margin

print("-" * 50)
print(f"🎯 Calculated Optimal OOF Threshold: {best_threshold:.4f}")
print(f"🎯 Applied Safety Threshold: {final_threshold:.4f}")
print("-" * 50)

--------------------------------------------------
🎯 Calculated Optimal OOF Threshold: 0.0023
🎯 Applied Safety Threshold: 0.0100
--------------------------------------------------


In [56]:
# 6. Retrain on Full Dataset for Final Predictions
print("Retraining ensemble on full dataset...")
test_probs = np.zeros((len(X_test_sc), len(models)))

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train_sc, y_train)
    test_probs[:, i] = model.predict_proba(X_test_sc)[:, 1]

# Average the final test probabilities
ensemble_test_probs = test_probs.mean(axis=1)
test_preds = (ensemble_test_probs >= final_threshold).astype(int)

Retraining ensemble on full dataset...


In [57]:
# 7. Save Submission
submission = pd.DataFrame({
    'CoilID': test_ids,
    'Y': test_preds
})

submission.to_csv('expected_submission_final.csv', index=False)
print("✅ Submission saved as 'expected_submission_final.csv'")
print(f"🔥🔥 Total defects predicted in test set: {test_preds.sum()} out of {len(test_preds)} (Target: ~26) 🔥🔥")

✅ Submission saved as 'expected_submission_final.csv'
🔥🔥 Total defects predicted in test set: 141 out of 339 (Target: ~26) 🔥🔥
